# rna-state-inf
<!-- SPDX-License-Identifier: GPL-3.0-only -->

Adapted from `sinc-lab/lncRNA-folding`.
Modified by Jingwen Liu, 2026, for full-length viral RNA benchmarking.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import pandas as pd
import time
from pathlib import Path

import numpy as np
import tensorflow as tf

# disable warning TF logs
tf.get_logger().setLevel('ERROR')

In [ ]:
method_name  = "rna-state-inf"
base = Path.cwd()

tools_dir = base.parent / 'tools'
tools_dir.mkdir(exist_ok=True)

model_fname  = 'rna-state-inf_rnnepoch50.h5'
gtfold_fname = 'gtfold-3.0_x86-64_ubuntu'

url1 = f"https://raw.githubusercontent.com/sinc-lab/lncRNA-folding/main/methods/{model_fname}"
url2 = f"https://master.dl.sourceforge.net/project/gtfold/{gtfold_fname}.tar.gz?viasf=1"

In [ ]:
# download trained model used for estimating SHAPE data
os.chdir(tools_dir)
if not Path(model_fname).exists():
    !wget -q {url1}
else:
    print(f"Model already exists: {model_fname}")

In [ ]:
# Pre-load the TensorFlow model
model_file_path = tools_dir / model_fname
loaded_model = tf.keras.models.load_model(str(model_file_path))

In [ ]:
# download method to predict secondary structure using SHAPE data
gtfold_dir = tools_dir / gtfold_fname
if not gtfold_dir.exists():
    !wget -q {url2} -O {gtfold_fname}.tar.gz
    !tar xfz {gtfold_fname}.tar.gz
else:
    print(f"GTFold already exists: {gtfold_dir}")

gfold_data = str(gtfold_dir / 'data')
gfold_bin  = str(gtfold_dir / 'gtmfe')

!{gfold_bin} | head -3

In [ ]:
os.chdir(base)
prediction_dir = base.parent / 'prediction'
prediction_dir.mkdir(exist_ok=True)

In [ ]:
def read_virus_fasta(path: str):
    lines = [ln.strip() for ln in open(path, 'r').read().splitlines() if ln.strip() != '']
    records = []
    for i in range(0, len(lines), 3):
        header, seq, struct = lines[i], lines[i+1], lines[i+2]
        name = header[1:].strip()
        records.append((name, seq.strip(), struct.strip()))
    df = pd.DataFrame(records, columns=['name','sequence','structure']).set_index('name')
    return df

viruses = read_virus_fasta('../data/viruses.fasta')

selected_virus_keys = None

if selected_virus_keys is None:
    virus_ids = list(viruses.index)
else:
    tmp = []
    for k in selected_virus_keys:
        if isinstance(k, int):
            tmp.append(viruses.index[k])
        else:
            tmp.append(str(k))
    virus_ids = tmp

In [ ]:
#@title Utils
sequencedict = {'A' : 0, 'C' : 1, 'G' : 2, 'U' : 3}
for letter in 'BDEFHIJKLMNOPQRSTVWXYZ':
    sequencedict.update({letter : 4})

def process_sequence(fin):
    with open(fin) as f:
        for a in f:
            _, seq, _ = a.strip().split('\t')

    seq = [sequencedict[s] for s in list(seq)]
    seq = tf.one_hot(seq, 5)
    seq = tf.convert_to_tensor([seq])

    return seq

def prob2shape(lstm_probs):
    """This method was taken from the following link: 
    https://github.com/dwillmott/rna-state-inf/blob/master/method.py
    """
    a = 0.214
    b = 0.6624
    s = 0.3603

    outstr = ""
    #for each position, write a SHAPE value depending on the probability
    for i, lstm_prob in enumerate(lstm_probs):
        if lstm_prob >= 0.5:
            shapevalue = ((a-s)/0.5)*(lstm_prob-1)+a
        else:
            shapevalue = ((s-b)/0.5)*lstm_prob+b

        outstr += '%d %.3f ' % (i+1, shapevalue)
        outstr += '\n'

    return outstr

def make_prediction(seq_fin, model_fin, outfname):
    seq   = process_sequence(seq_fin)
    model = tf.keras.models.load_model(model_fin)

    pred  = model.predict(seq, verbose=0)
    strout = prob2shape(pred[0,:,1])

    fout = open(outfname, 'w')
    fout.write(strout)
    fout.close()

def fas2tsv(fin):
  seq = ""
  with open(fin) as f:
    for a in f:
      b = a.strip()
      if b[0] == ">":
        desc = b[1:]
      else:
        seq += b
  
  # write files
  foutname = fin + '.tsv'
  out = open(foutname, 'w')
  out.write('{}\t{}\t{}\n'.format(desc, seq, desc))
  out.close()

  foutseq = fin + '.seq'
  out = open(foutseq, 'w')
  out.write('{}\n'.format(seq))
  out.close()

  return foutname, foutseq

In [ ]:
def run_folding(fasta_name, model_file, output_dir):
  out_file_name = fasta_name + '.dot'
  tmp_file1 = out_file_name + '.tmp1'
  tmp_file2 = out_file_name + '.tmp2'

  tsv_file, seq_file = fas2tsv(fasta_name)

  # estimate SHAPE-like data
  make_prediction(tsv_file, model_file, tmp_file1)
  
  # predict structure from estimated SHAPE data
  os.system(f"export GTFOLDDATADIR={gfold_data}; {gfold_bin} {seq_file} --useSHAPE {tmp_file1} --output {out_file_name} > /dev/null 2>&1")
  
  # convert ct to dot-bracket format using custom ct2dot.py
  ct_file = out_file_name + '.ct'
  os.system(f"python ct2dot.py {ct_file} {tmp_file2} -f full -q")

  # fix description line
  os.system(f"head -1 {fasta_name}   > {out_file_name}")
  os.system(f"tail -n+2 {tmp_file2} >> {out_file_name}")

  final_output = str(output_dir / Path(out_file_name).name)
  os.system(f"cp {out_file_name} {final_output}")
  
  for tmp_file in [fasta_name, tsv_file, seq_file, tmp_file1, tmp_file2, out_file_name, ct_file]:
    if os.path.exists(tmp_file):
      os.remove(tmp_file)
  
  return final_output

In [ ]:
out_fasta_name = method_name
final_fasta_path = prediction_dir / (out_fasta_name + ".fasta")
if final_fasta_path.exists(): 
    final_fasta_path.unlink()

print(f"{' ':3}\t{'virus':<20}\t{'len':<5}\t{'time'}")
for i, vid in enumerate(virus_ids):

  start_time = time.time()
  seq = viruses.loc[vid]["sequence"]
  print(f"{i+1:3d}/{len(virus_ids)}\t{vid:<20}\t{len(seq):<5}\t", end='', flush=True)

  # Write a one-sequence fasta
  with open("tmp.fasta", "w") as ofile: 
    ofile.write(f">{vid}\n{seq}\n")
  
  model_file_path = str(tools_dir / model_fname)
  dot_file_name = run_folding("tmp.fasta", model_file_path, prediction_dir)

  # Concatenate outputs
  os.system(f"cat {dot_file_name} >> {final_fasta_path}") 

  print(f"{time.time() - start_time: .1f} s")